# Análisis de datos composicionales de microbioma: CLR, PCA, beta diversidad y clustering

Este notebook toma la tabla de abundancias y los metadatos del repositorio, trata los datos como **composicionales**, aplica transformación **CLR** y calcula:

- PCA sobre datos CLR
- Beta diversidad usando distancia Euclidiana sobre CLR
- PCoA desde la matriz de distancia
- Clustering jerárquico
- K-means exploratorio

Todas las figuras se guardan en la carpeta `plots/` y las tablas en `results/`.

In [1]:
# ===============================
# 1. Librerías y configuración
# ===============================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Carpetas de salida
plots_dir = Path("plots")
results_dir = Path("results")
plots_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300

In [28]:
# ===============================
# 2. URLs de entrada
# ===============================

ABUNDANCE_URL = "https://raw.githubusercontent.com/shadayguerrero/neonatal-gut-microbiome-analysis/refs/heads/main/data/all_samples.tsv"
METADATA_URL = "https://raw.githubusercontent.com/shadayguerrero/neonatal-gut-microbiome-analysis/refs/heads/main/data/metadata_cibimi.csv"

# Parámetros ajustables
# Si sabes cuál columna identifica las muestras en metadata, ponla aquí. Si no, deja None.
sample_id_col = "MBI Sample ID"

# Columna para colorear los gráficos. Si no sabes, deja None y el notebook intentará escoger una.
color_col = None

# Filtrado de taxones antes del CLR
min_prevalence = 0.05     # Mantener taxones presentes en al menos 5% de muestras
min_mean_abundance = 0.0  # Cambia a 0.0001 o 0.001 si hay demasiados taxones
max_taxa = 200            # Mantener los 200 taxones más abundantes; cambia a None para no limitar

# Clustering
kmeans_min_k = 2
kmeans_max_k = 6

In [29]:
# ===============================
# 3. Cargar datos
# ===============================

abund_raw = pd.read_csv(ABUNDANCE_URL, delim_whitespace=True, skiprows=1, header=0, on_bad_lines='skip')
metadata = pd.read_csv(METADATA_URL)

print("Abundancias crudas:", abund_raw.shape)
print("Metadata:", metadata.shape)

display(abund_raw.head())
display(metadata.head())

Abundancias crudas: (7002, 110)
Metadata: (86, 16)


,,,,#OTU,ID,DNA-3OVK_S2101_R-taxa,DNA-3OWT_S2551_R-taxa,DNA-3OXF_S2771_R-taxa,S001Z-0004_S2501_R-taxa,S001Z-0007_S1661_R-taxa,S001Z-0007_S2811_R-taxa,S001Z-0007_S5342_R-taxa,S001Z-0007_S941_R-taxa,...,S00SO-0078_S2701_R-taxa,S00SO-0079_S2711_R-taxa,S00SO-0080_S2721_R-taxa,S00SO-0081_S2731_R-taxa,S00SO-0082_S2741_R-taxa,S00SO-0083_S2751_R-taxa,S00SO-0084_S2761_R-taxa,S00SO-0086_S2781_R-taxa,Consensus,Lineage
2,28.0,23.0,58.0,16.0,48.0,34.0,55.0,50.0,181.0,49.0,50.0,35.0,4456.0,...,47.0,247.0,4735.0,k__Bacteria;,p__;,c__;,o__;,f__;,g__;,s__
976,11.0,9.0,14.0,9.0,17.0,11.0,11.0,10.0,21.0,25.0,7.0,11.0,20.0,...,17.0,9.0,44.0,k__Bacteria;,p__Bacteroidota;,c__;,o__;,f__;,g__;,s__
171549,64.0,77.0,103.0,59.0,106.0,63.0,38.0,80.0,466.0,124.0,79.0,101.0,87.0,...,169.0,86.0,269.0,k__Bacteria;,p__Bacteroidota;,c__Bacteroidia;,o__Bacteroidales;,f__;,g__;,s__
815,17.0,20.0,34.0,15.0,42.0,18.0,31.0,32.0,198.0,35.0,21.0,42.0,30.0,...,38.0,19.0,100.0,k__Bacteria;,p__Bacteroidota;,c__Bacteroidia;,o__Bacteroidales;,f__Bacteroidaceae;,g__;,s__
816,100.0,96.0,120.0,112.0,172.0,110.0,68.0,119.0,655.0,277.0,81.0,143.0,134.0,...,211.0,160.0,284.0,k__Bacteria;,p__Bacteroidota;,c__Bacteroidia;,o__Bacteroidales;,f__Bacteroidaceae;,g__Bacteroides;,s__


,MBI Sample ID,Your Sample ID,Baby number,Sampling Date,Weeks of gestation,Sample Type,Group,Sex,Weight,Size,Head Circunference,Abdominal Circunference,Necrotizing enterocolitis,Type of Birth,Clasification birth weight,Week
0,S00SO-0001,C3.1,3,Week 1 (Birth),27.0,human feces,CONTROL,fem,800.0,34.0,24.0,18.0,No presence,Cesarean,Extremely low,1
1,S00SO-0002,C3.2,3,Week 2,NaN,human feces,CONTROL,fem,805.0,34.0,23.0,18.0,No presence,Cesarean,Extremely low,2
2,S00SO-0003,C3.3,3,Week 3,NaN,human feces,CONTROL,fem,870.0,34.0,24.0,20.0,No presence,Cesarean,Extremely low,3
3,S00SO-0004,C3.4,3,Week 4,NaN,human feces,CONTROL,fem,910.0,34.0,24.0,20.0,No presence,Cesarean,Extremely low,4
4,S00SO-0005,C3.5,3,Week 5,NaN,human feces,CONTROL,fem,NaN,NaN,NaN,NaN,No presence,Cesarean,Extremely low,5


In [30]:
# ===============================
# 4. Funciones auxiliares
# ===============================

def make_index_if_needed(df):
    """Usa la primera columna como índice si parece ser identificador de taxón/muestra."""
    df = df.copy()
    first_col = df.columns[0]
    # If the first column name is 'OTU ID', or if less than 50% are numeric, assume it's an ID
    numeric_first = pd.to_numeric(df[first_col], errors="coerce").notna().mean()
    if first_col == 'OTU ID' or numeric_first < 0.5:
        df = df.set_index(first_col)
    return df

def normalize_abundance_sample_names(abundance_df_columns):
    """
    Extracts the base sample ID from complex abundance column names.
    e.g., 'S00SO-0001_S1931_R-taxa' -> 'S00SO-0001'
    """
    normalized_names = []
    for col_name in abundance_df_columns:
        col_name_str = str(col_name)
        # Assuming format like SXXXX-XXXX_YYYY or DNA-XXXX_YYYY or S001Z-XXXX_YYYY
        if '_' in col_name_str:
            normalized_names.append(col_name_str.split('_')[0])
        else:
            normalized_names.append(col_name_str)
    return normalized_names

def prepare_abundance_matrix(abund_raw, metadata, sample_id_col):
    """
    Devuelve matriz con muestras en filas y taxones/features en columnas,
    filtrando y normalizando las muestras.
    """
    df = make_index_if_needed(abund_raw)
    df.index = df.index.astype(str)
    df.columns = df.columns.astype(str)

    # Normalize abundance column names to match metadata IDs
    df.columns = normalize_abundance_sample_names(df.columns)

    # Convert to numeric, dropping non-numeric columns/rows
    numeric_df = df.apply(pd.to_numeric, errors="coerce")
    numeric_df = numeric_df.dropna(axis=1, how="all") # Drop columns that are all NaN (non-numeric features)
    numeric_df = numeric_df.dropna(axis=0, how="all") # Drop rows that are all NaN (e.g. if original index was numeric and converted to NaN)

    # Get sample IDs from metadata
    if sample_id_col not in metadata.columns:
        raise ValueError(f"La columna '{sample_id_col}' no se encontró en los metadatos.")

    metadata_sample_ids = set(metadata[sample_id_col].dropna().astype(str))

    # Filter abundance matrix columns (samples) based on metadata samples
    # Assuming samples are currently in columns after make_index_if_needed and normalization
    samples_in_common = [col for col in numeric_df.columns if col in metadata_sample_ids]
    mat = numeric_df[samples_in_common]

    # Transpose to have samples in rows and features in columns
    mat = mat.T

    mat.index = mat.index.astype(str)
    mat.columns = mat.columns.astype(str)
    mat = mat.fillna(0)
    mat = mat.loc[mat.sum(axis=1) > 0, mat.sum(axis=0) > 0] # Remove samples/features with zero total abundance

    print(f"Muestras retenidas después de filtrar por metadata: {mat.shape[0]}")
    return mat, sample_id_col

def choose_color_column(metadata, sample_id_col):
    """Escoge una columna categórica útil para colorear si el usuario no define una."""
    candidates = []
    for col in metadata.columns:
        if col == sample_id_col:
            continue
        nunique = metadata[col].nunique(dropna=True)
        if 2 <= nunique <= 12:
            candidates.append((col, nunique))
    return candidates[0][0] if candidates else None


def multiplicative_replacement(X, delta=None):
    """
    Reemplazo de ceros para datos composicionales.
    X debe estar cerrado por filas, es decir cada fila suma 1.
    """
    X = np.asarray(X, dtype=float)
    X_repl = X.copy()
    positive = X_repl[X_repl > 0]
    if delta is None:
        delta = positive.min() / 2

    D = X_repl.shape[1]
    for i in range(X_repl.shape[0]):
        row = X_repl[i, :]
        zero_mask = row == 0
        z = zero_mask.sum()
        if z == 0:
            continue
        # Asegurar que la masa agregada a ceros no supere 65% de la composición
        d = min(delta, 0.65 / max(z, 1))
        positive_mask = ~zero_mask
        positive_sum = row[positive_mask].sum()
        row[zero_mask] = d
        row[positive_mask] = row[positive_mask] * (1 - z * d) / positive_sum
        X_repl[i, :] = row
    return X_repl


def clr_transform(X):
    """Centered log-ratio: log(x) - media(log(x)) por muestra."""
    X = np.asarray(X, dtype=float)
    logX = np.log(X)
    return logX - logX.mean(axis=1, keepdims=True)


def pcoa_from_distance(D, n_components=2):
    """PCoA clásica desde una matriz de distancia."""
    D = np.asarray(D, dtype=float)
    n = D.shape[0]
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ (D ** 2) @ J
    eigvals, eigvecs = np.linalg.eigh(B)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    positive = eigvals > 0
    eigvals_pos = eigvals[positive]
    eigvecs_pos = eigvecs[:, positive]
    coords = eigvecs_pos[:, :n_components] * np.sqrt(eigvals_pos[:n_components])
    explained = eigvals_pos / eigvals_pos.sum()
    return coords, explained[:n_components], eigvals

In [16]:
# ===============================
# 5. Preparar matriz composicional
# ===============================

abundance, sample_id_col = prepare_abundance_matrix(abund_raw, metadata, sample_id_col)

print("Matriz con muestras en filas y taxones/features en columnas:", abundance.shape)
display(abundance.iloc[:5, :5])

# Normalizar/cerrar a abundancia relativa por muestra
composition = abundance.div(abundance.sum(axis=1), axis=0)
composition = composition.replace([np.inf, -np.inf], np.nan).fillna(0)

# Filtrar taxones/features raros
prevalence = (composition > 0).mean(axis=0)
mean_abundance = composition.mean(axis=0)
keep = (prevalence >= min_prevalence) & (mean_abundance >= min_mean_abundance)
composition_filt = composition.loc[:, keep]

# Mantener top N por abundancia media si se definió max_taxa
if max_taxa is not None and composition_filt.shape[1] > max_taxa:
    top_cols = composition_filt.mean(axis=0).sort_values(ascending=False).head(max_taxa).index
    composition_filt = composition_filt.loc[:, top_cols]

# Cerrar nuevamente después del filtrado
composition_filt = composition_filt.div(composition_filt.sum(axis=1), axis=0).fillna(0)

print("Matriz filtrada:", composition_filt.shape)
print("Taxones/features retenidos:", composition_filt.shape[1])

composition_filt.to_csv(results_dir / "composition_relative_abundance_filtered.csv")

TypeError: Setting a MultiIndex dtype to anything other than object is not supported

In [17]:
# ===============================
# 6. Reemplazo de ceros + transformación CLR
# ===============================

if composition_filt.empty:
    print("composition_filt está vacío. No se puede realizar el reemplazo de ceros ni la transformación CLR.")
    clr_df = pd.DataFrame(index=composition_filt.index, columns=composition_filt.columns)
else:
    X_comp = composition_filt.values
    X_repl = multiplicative_replacement(X_comp)
    X_clr = clr_transform(X_repl)

    clr_df = pd.DataFrame(X_clr, index=composition_filt.index, columns=composition_filt.columns)
    clr_df.to_csv(results_dir / "clr_transformed_abundance.csv")

print("Datos CLR:", clr_df.shape)
display(clr_df.iloc[:5, :5])

composition_filt está vacío. No se puede realizar el reemplazo de ceros ni la transformación CLR.
Datos CLR: (0, 0)


""
# Constructed from biom file


In [18]:
# ===============================
# 7. Unir con metadata
# ===============================

if sample_id_col is not None:
    metadata2 = metadata.copy()
    metadata2[sample_id_col] = metadata2[sample_id_col].astype(str)
    metadata2 = metadata2.set_index(sample_id_col)
    common_samples = clr_df.index.intersection(metadata2.index)
    print("Muestras con metadata:", len(common_samples), "de", clr_df.shape[0])
else:
    metadata2 = pd.DataFrame(index=clr_df.index)
    common_samples = clr_df.index

clr_df = clr_df.loc[common_samples]
composition_filt = composition_filt.loc[common_samples]
metadata2 = metadata2.loc[common_samples] if len(common_samples) > 0 else metadata2

if color_col is None and metadata2.shape[1] > 0:
    color_col = choose_color_column(metadata2.reset_index(), metadata2.index.name)

print("Columna usada para color:", color_col)
if color_col is not None:
    print(metadata2[color_col].value_counts(dropna=False).head(20))

Muestras con metadata: 0 de 0
Columna usada para color: Sampling Date
Sampling Date
Week 1 (Birth)    25
Week 2            24
Week 3            19
Week 4            12
Week 5             6
Name: count, dtype: int64


In [19]:
# ===============================
# 8. PCA sobre CLR
# ===============================

if clr_df.empty or clr_df.shape[0] < 2:
    print("clr_df está vacío o tiene muy pocas muestras para PCA.")
    pca_df = pd.DataFrame(columns=["SampleID", "PC1", "PC2"])
    loadings_df = pd.DataFrame()
    pca_explained_variance_ratio = [0, 0]
else:
    pca = PCA(n_components=2)
    pca_coords = pca.fit_transform(clr_df.values)

    pca_df = pd.DataFrame({
        "SampleID": clr_df.index,
        "PC1": pca_coords[:, 0],
        "PC2": pca_coords[:, 1],
    })
    if color_col is not None and color_col in metadata2.columns:
        pca_df[color_col] = metadata2[color_col].values

    loadings_df = pd.DataFrame(
        pca.components_.T,
        index=clr_df.columns,
        columns=["PC1_loading", "PC2_loading"]
    )
    pca_explained_variance_ratio = pca.explained_variance_ratio_

pca_df.to_csv(results_dir / "pca_clr_scores.csv", index=False)
loadings_df.to_csv(results_dir / "pca_clr_loadings.csv")

print("Varianza explicada PC1 y PC2:", pca_explained_variance_ratio)
display(pca_df.head())

clr_df está vacío o tiene muy pocas muestras para PCA.
Varianza explicada PC1 y PC2: [0, 0]


,SampleID,PC1,PC2


In [20]:
# Graficar PCA

if not pca_df.empty and pca_df.shape[0] > 1:
    fig, ax = plt.subplots(figsize=(7, 5))

    if color_col is not None and color_col in pca_df.columns:
        groups = pca_df[color_col].astype(str).fillna("NA")
        for group in sorted(groups.unique()):
            ix = groups == group
            ax.scatter(pca_df.loc[ix, "PC1"], pca_df.loc[ix, "PC2"], label=group, alpha=0.8)
        ax.legend(title=color_col, bbox_to_anchor=(1.05, 1), loc="upper left")
    else:
        ax.scatter(pca_df["PC1"], pca_df["PC2"], alpha=0.8)

    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_xlabel(f"PC1 ({pca_explained_variance_ratio[0]*100:.1f} %)")
    ax.set_ylabel(f"PC2 ({pca_explained_variance_ratio[1]*100:.1f} %)")
    ax.set_title("PCA sobre abundancias transformadas con CLR")
    plt.tight_layout()
    plt.savefig(plots_dir / "pca_clr.png", bbox_inches="tight")
    plt.show()
else:
    print("No hay datos suficientes para graficar PCA.")

No hay datos suficientes para graficar PCA.


In [21]:
# ===============================
# 9. Beta diversidad: Euclidiana sobre CLR
# ===============================

if clr_df.empty or clr_df.shape[0] < 2:
    print("clr_df está vacío o tiene muy pocas muestras para calcular la distancia euclidiana.")
    dist_df = pd.DataFrame()
else:
    D = squareform(pdist(clr_df.values, metric="euclidean"))
    dist_df = pd.DataFrame(D, index=clr_df.index, columns=clr_df.index)
    dist_df.to_csv(results_dir / "beta_diversity_euclidean_on_clr_distance_matrix.csv")

print("Matriz de distancia:", dist_df.shape)
display(dist_df.iloc[:5, :5])

clr_df está vacío o tiene muy pocas muestras para calcular la distancia euclidiana.
Matriz de distancia: (0, 0)


""


In [22]:
# ===============================
# 10. PCoA desde distancia Euclidiana sobre CLR
# ===============================

if dist_df.empty:
    print("No hay matriz de distancia para PCoA.")
    pcoa_df = pd.DataFrame(columns=["SampleID", "PCoA1", "PCoA2"])
    pcoa_explained = [0, 0]
else:
    pcoa_coords, pcoa_explained, eigvals = pcoa_from_distance(D, n_components=2)

    pcoa_df = pd.DataFrame({
        "SampleID": clr_df.index,
        "PCoA1": pcoa_coords[:, 0],
        "PCoA2": pcoa_coords[:, 1],
    })
    if color_col is not None and color_col in metadata2.columns:
        pcoa_df[color_col] = metadata2[color_col].values

pcoa_df.to_csv(results_dir / "pcoa_euclidean_on_clr_scores.csv", index=False)

print("Varianza explicada PCoA1 y PCoA2:", pcoa_explained)
display(pcoa_df.head())

No hay matriz de distancia para PCoA.
Varianza explicada PCoA1 y PCoA2: [0, 0]


,SampleID,PCoA1,PCoA2


In [23]:
# Graficar PCoA
if not pcoa_df.empty and pcoa_df.shape[0] > 1:
    fig, ax = plt.subplots(figsize=(7, 5))

    if color_col is not None and color_col in pcoa_df.columns:
        groups = pcoa_df[color_col].astype(str).fillna("NA")
        for group in sorted(groups.unique()):
            ix = groups == group
            ax.scatter(pcoa_df.loc[ix, "PCoA1"], pcoa_df.loc[ix, "PCoA2"], label=group, alpha=0.8)
        ax.legend(title=color_col, bbox_to_anchor=(1.05, 1), loc="upper left")
    else:
        ax.scatter(pcoa_df["PCoA1"], pcoa_df["PCoA2"], alpha=0.8)

    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_xlabel(f"PCoA1 ({pcoa_explained[0]*100:.1f} %)")
    ax.set_ylabel(f"PCoA2 ({pcoa_explained[1]*100:.1f} %)")
    ax.set_title("PCoA - distancia Euclidiana sobre CLR")
    plt.tight_layout()
    plt.savefig(plots_dir / "pcoa_euclidean_on_clr.png", bbox_inches="tight")
    plt.show()
else:
    print("No hay datos suficientes para graficar PCoA.")

No hay datos suficientes para graficar PCoA.


In [24]:
# ===============================
# 11. Clustering jerárquico
# ===============================

if clr_df.empty or clr_df.shape[0] < 2:
    print("clr_df está vacío o tiene muy pocas muestras para clustering jerárquico.")
    hier_df = pd.DataFrame(columns=["SampleID", "hierarchical_cluster"])
else:
    Z = linkage(clr_df.values, method="ward", metric="euclidean")

    # Definir clusters jerárquicos. Cambia t si quieres otro número de clusters.
    hierarchical_k = 3
    hier_clusters = fcluster(Z, t=hierarchical_k, criterion="maxclust")

    hier_df = pd.DataFrame({
        "SampleID": clr_df.index,
        "hierarchical_cluster": hier_clusters
    })
    hier_df.to_csv(results_dir / "hierarchical_clusters_clr.csv", index=False)

    fig, ax = plt.subplots(figsize=(12, 5))
    dendrogram(Z, labels=clr_df.index.tolist(), leaf_rotation=90, ax=ax)
    ax.set_title("Clustering jerárquico sobre CLR - método Ward")
    ax.set_ylabel("Distancia")
    plt.tight_layout()
    plt.savefig(plots_dir / "hierarchical_clustering_dendrogram_clr.png", bbox_inches="tight")
    plt.show()

display(hier_df.head())

clr_df está vacío o tiene muy pocas muestras para clustering jerárquico.


,SampleID,hierarchical_cluster


In [25]:
# ===============================
# 12. K-means exploratorio sobre CLR
# ===============================

if clr_df.empty or clr_df.shape[0] < 2:
    print("clr_df está vacío o tiene muy pocas muestras para K-means.")
    sil_df = pd.DataFrame(columns=["k", "silhouette_score"])
    kmeans_labels = np.array([])
else:
    silhouette_results = []
    max_possible_k = min(kmeans_max_k, clr_df.shape[0] - 1)

    if max_possible_k >= kmeans_min_k:
        for k in range(kmeans_min_k, max_possible_k + 1):
            km = KMeans(n_clusters=k, random_state=42, n_init=20)
            labels = km.fit_predict(clr_df.values)
            sil = silhouette_score(clr_df.values, labels, metric="euclidean")
            silhouette_results.append((k, sil))

    sil_df = pd.DataFrame(silhouette_results, columns=["k", "silhouette_score"])
sil_df.to_csv(results_dir / "kmeans_silhouette_scores_clr.csv", index=False)

if len(sil_df) > 0:
    best_k = int(sil_df.sort_values("silhouette_score", ascending=False).iloc[0]["k"])
else:
    best_k = 2

print("Mejor k por silhouette:", best_k)
display(sil_df)

if clr_df.empty or clr_df.shape[0] < best_k:
    print("No hay suficientes muestras para ejecutar K-means con el mejor k.")
    kmeans_labels = np.array([])
else:
    km = KMeans(n_clusters=best_k, random_state=42, n_init=20)
    kmeans_labels = km.fit_predict(clr_df.values) + 1

kmeans_df = pd.DataFrame({
    "SampleID": clr_df.index,
    "kmeans_cluster": kmeans_labels
})
kmeans_df.to_csv(results_dir / "kmeans_clusters_clr.csv", index=False)

clr_df está vacío o tiene muy pocas muestras para K-means.
Mejor k por silhouette: 2


,k,silhouette_score


No hay suficientes muestras para ejecutar K-means con el mejor k.


In [26]:
# Graficar silhouette por k
if len(sil_df) > 0 and not sil_df.isnull().values.any(): # Added check for null values
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(sil_df["k"], sil_df["silhouette_score"], marker="o")
    ax.set_xlabel("Número de clusters k")
    ax.set_ylabel("Silhouette score")
    ax.set_title("Selección exploratoria de k para K-means sobre CLR")
    plt.tight_layout()
    plt.savefig(plots_dir / "kmeans_silhouette_clr.png", bbox_inches="tight")
    plt.show()
else:
    print("No hay datos suficientes o válidos para graficar el score silhouette.")

No hay datos suficientes o válidos para graficar el score silhouette.


In [27]:
# PCA coloreado por cluster K-means

if not pca_df.empty and len(kmeans_labels) == len(pca_df) and pca_df.shape[0] > 1:
    pca_cluster_df = pca_df.copy()
    pca_cluster_df["kmeans_cluster"] = kmeans_labels.astype(str)

    fig, ax = plt.subplots(figsize=(7, 5))
    for group in sorted(pca_cluster_df["kmeans_cluster"].unique()):
        ix = pca_cluster_df["kmeans_cluster"] == group
        ax.scatter(pca_cluster_df.loc[ix, "PC1"], pca_cluster_df.loc[ix, "PC2"], label=f"Cluster {group}", alpha=0.8)

    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_xlabel(f"PC1 ({pca_explained_variance_ratio[0]*100:.1f} %)")
    ax.set_ylabel(f"PC2 ({pca_explained_variance_ratio[1]*100:.1f} %)")
    ax.set_title("PCA sobre CLR coloreado por K-means")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(plots_dir / "pca_clr_kmeans_clusters.png", bbox_inches="tight")
    plt.show()
else:
    print("No hay datos suficientes para graficar PCA coloreado por clusters de K-means.")

No hay datos suficientes para graficar PCA coloreado por clusters de K-means.


In [ ]:
# ===============================
# 13. Resumen de archivos generados
# ===============================

print("Figuras guardadas en:", plots_dir.resolve())
for f in sorted(plots_dir.glob("*.png")):
    print(" -", f)

print("\nTablas guardadas en:", results_dir.resolve())
for f in sorted(results_dir.glob("*.csv")):
    print(" -", f)

## Notas importantes

- Para gráficos de composición o abundancia relativa, usa la matriz `composition_relative_abundance_filtered.csv`, no la matriz CLR.
- Para PCA, distancia Euclidiana, PCoA y clustering, usa `clr_transformed_abundance.csv`.
- La distancia Euclidiana después de CLR corresponde a una aproximación práctica de la distancia de Aitchison para datos composicionales.
- Si hay demasiados taxones, aumenta `min_prevalence`, `min_mean_abundance` o baja `max_taxa`.